# ENSAMBLE DE LOS 3 MODELOS (SOFT)

In [ ]:
import re
import math
import numpy as np
import pandas as pd
from collections import Counter
import joblib                         

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV  
from sklearn.preprocessing import StandardScaler


# ═════════════════════════════════════════════════════════════════════════════
# SÍMBOLOS Y EXPRESIONES REGULARES
# ═════════════════════════════════════════════════════════════════════════════

# Símbolos tipográficos comunes que el Random Forest usa como features numéricas
SIMBOLOS_FRECUENTES = [
    ',', '.', '-', '^', ':', ';', "'", '*', '(', '»',
    '/', '?', ')', '¿', '>', '«', '¡', '!', '|', '&',
    '¬', '<', '£', '"', '—', '$', '•', '\\', 'ſ', '[',
    '=', '\u2018', ']', '%', '■', '\u201c', '©', '}', '€', '{',
    '°', '„', '#', '+', '®', '\u201d', '\u2019', '_'
]
SIMBOLOS_MEDIEVALES = ['ꝛ', '§', '~', '⁊', 'ẽ', 'ũ', '⸗', 'õ', '™', 'ã', '♦']  # Símbolos medievales raros pero discriminativos para textos muy antiguos
TODOS_SIMBOLOS      = SIMBOLOS_FRECUENTES + SIMBOLOS_MEDIEVALES               # Lista completa de 59 símbolos

def nombre_columna(simbolo):
    # Convierte cada carácter en un nombre de columna
    nombres = {
        ',': 'sim_coma', '.': 'sim_punto', '-': 'sim_guion', '^': 'sim_caret',
        ':': 'sim_dos_puntos', ';': 'sim_punto_coma', "'": 'sim_apostrofe', '*': 'sim_asterisco',
        '(': 'sim_paren_abrir', '»': 'sim_guillemet_der', '/': 'sim_barra', '?': 'sim_interrogacion',
        ')': 'sim_paren_cerrar', '¿': 'sim_interrogacion_es', '>': 'sim_mayor', '«': 'sim_guillemet_izq',
        '¡': 'sim_exclamacion_es', '!': 'sim_exclamacion', '|': 'sim_barra_vertical', '&': 'sim_ampersand',
        '¬': 'sim_silabeo', '<': 'sim_menor', '£': 'sim_libra', '"': 'sim_comilla_doble',
        '—': 'sim_raya', '$': 'sim_dolar', '•': 'sim_viñeta', '\\': 'sim_barra_inversa',
        'ſ': 'sim_s_larga', '[': 'sim_corchete_abrir', '=': 'sim_igual', '\u2018': 'sim_comilla_izq',
        ']': 'sim_corchete_cerrar', '%': 'sim_porcentaje', '■': 'sim_cuadrado', '\u201c': 'sim_comilla_doble_izq',
        '©': 'sim_copyright', '}': 'sim_llave_cerrar', '€': 'sim_euro', '{': 'sim_llave_abrir',
        '°': 'sim_grado', '„': 'sim_comilla_baja', '#': 'sim_almohadilla', '+': 'sim_mas',
        '®': 'sim_marca_reg', '\u201d': 'sim_comilla_doble_der', '\u2019': 'sim_comilla_der', '_': 'sim_guion_bajo',
        'ꝛ': 'sim_r_medieval', '§': 'sim_seccion', '~': 'sim_tilde', '⁊': 'sim_tironian_et',
        'ẽ': 'sim_e_tilde', 'ũ': 'sim_u_tilde', '⸗': 'sim_guion_doble', 'õ': 'sim_o_tilde',
        '™': 'sim_trademark', 'ã': 'sim_a_tilde', '♦': 'sim_rombo',
    }
    return nombres.get(simbolo, f'sim_{ord(simbolo)}')

COLUMNAS_SIMBOLOS = [nombre_columna(s) for s in TODOS_SIMBOLOS]   # Lista de nombres de columna para los 59 símbolos

PALABRAS_ARCAICAS = re.compile(      # Vocabulario del español medieval y renacentista
    r'\b(?:efto|efta|eftos|eftas|dize|diziendo|dixo|hazer|fazer|dezir|cofa|'
    r'cofas|mifmo|mifma|afsi|affi|tambien|efpañ\w*|efcriv\w*|efcrib\w*|'
    r'vueftra|vueflra|vueftro|merced|feñor|feñora|ombre|fobre|fiempre|'
    r'defpues|aunque|donde|quando|porque|agora|anfi|afi|anſi)\b', re.IGNORECASE
)
PALABRAS_LATINAS = re.compile(       # Palabras con terminaciones latinas (ibus, orum, onis, etc.)
    r'\b\w+(?:ibus|orum|arum|onis|atis|ens|antis|um|us|is|ae)\b', re.IGNORECASE
)
NUMEROS_ROMANOS = re.compile(        # Números romanos válidos — frecuentes en textos históricos
    r'\b(M{0,4}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3}))\b', re.IGNORECASE
)
ABREVIATURAS      = re.compile(r'\b[a-záéíóúüñ]{1,5}\.')                      # Palabras cortas seguidas de punto — abreviaturas
U_COMO_V          = re.compile(r'\bv[aeiouáéíóú]', re.IGNORECASE)             # Confusión u/v característica del español pre-moderno
J_POR_X           = re.compile(                                                # Alternancia j/x propia de la ortografía renacentista (dixo/dijo)
    r'\b(?:di[jx]o|ba[jx]o|me[jx]or|de[jx]ar|de[jx]ó|hi[jx]o|'
    r'mu[jx]er|tra[jx]|[jx]unto|[jx]ugar|[jx]uzgar|[jx]uicio)\b', re.IGNORECASE
)
MAYUS_MID_ORACION   = re.compile(r'(?<![.!?\n])\s+[A-ZÁÉÍÓÚÜÑ][a-záéíóúüñ]{2,}')  # Mayúsculas en mitad de oración — nombres propios o latinismos
CONECTORES_MODERNOS = re.compile(    # Conectores discursivos propios del español moderno (s.XVIII-XIX)
    r'\b(?:sin embargo|no obstante|es decir|por lo tanto|en consecuencia|'
    r'por consiguiente|a pesar de|dado que|puesto que|ya que|'
    r'en efecto|de hecho|por otro lado|en cambio|así pues)\b', re.IGNORECASE
)
VERBOS_ARCAICOS = re.compile(        # Desinencias verbales medievales como -edes, -ades, -istes
    r'\b\w+(?:edes|ades|istes|ábades|íades|áredes|éredes|íredes)\b', re.IGNORECASE
)
CHARS_AISLADOS = re.compile(r'(?<!\w)\b([b-df-hj-np-tv-zB-DF-HJ-NP-TV-Z])\b(?!\w)')  # Consonantes sueltas — basura común del OCR en manuscritos
F_INICIAL = re.compile(              # Palabras con f- inicial donde hoy va h- (fijo→hijo, fazer→hacer) — cambio ocurrido en s.XV-XVI
    r'\b(?:fab\w+|faz\w+|fij\w+|ferm\w+|ferr\w+|fiel\w+|fil\w+|'
    r'fin\w+|fue\w+|fuy\w+|fall\w+|fal\w+|fam\w+|far\w+|fat\w+|'
    r'fech\w+|fend\w+|ferv\w+|fest\w+|fid\w+|fog\w+|folg\w+|fond\w+|'
    r'font\w+|forc\w+|forn\w+|forr\w+|foy\w+|fuert\w+|fug\w+)\b', re.IGNORECASE
)

# Nombres de las 43 features estilométricas + los 59 nombres de símbolos
FEATURE_NAMES = [
    'caracteres', 'palabras', 'simbolos', 'mayusculas', 'oraciones',
    'longitud_promedio_palabra', 'ratio_doble_espacio', 'palabras_arcaicas',
    'palabras_latinas', 'ratio_vocales_acentuadas', 'signos_es',
    'comillas_angulares', 'guiones', 'ruido_ocr',
    'entropia', 'diversidad_lexica', 'ratio_palabras_cortas',
    'espacios_por_palabra', 'saltos_por_palabra', 'tabs_por_palabra',
    'longitud_promedio_linea', 'cortes_guion_fin_linea', 'digitos_por_palabra',
    'ratio_numeros_romanos', 'ratio_abreviaturas', 'ratio_u_como_v', 'ratio_j_por_x',
    'ratio_mayus_mid', 'densidad_parrafos',
    'ratio_conectores_modernos', 'ratio_verbos_arcaicos',
    'ratio_e_vs_y', 'ratio_non_vs_no', 'ratio_art_la',
    'ratio_chars_aislados', 'ratio_grupos_consonantes',
    'ratio_hapax', 'ratio_palabras_largas', 'ratio_oraciones_largas',
    'ratio_tildes_sobre_mayusculas', 'ratio_q', 'ratio_f_inicial', 'ratio_i_vs_y',
] + COLUMNAS_SIMBOLOS

N_FEATURES_ESTILO = 43
assert len(FEATURE_NAMES) == N_FEATURES_ESTILO + len(TODOS_SIMBOLOS)   # Verifica que el vector tenga exactamente 102 posiciones


# ═════════════════════════════════════════════════════════════════════════════
# FUNCIONES DE LIMPIEZA Y EXTRACCIÓN
# ═════════════════════════════════════════════════════════════════════════════

def clean_svc(texto):
    # Limpieza para el LinearSVC: convierte ñ→n porque el modelo fue entrenado así
    if not isinstance(texto, str): return ""
    texto = texto.lower()
    texto = texto.replace('ñ', 'n')                                        # La ñ se reemplaza para normalizar el vocabulario del TF-IDF
    texto = re.sub(r'[-¬]\s*', '', texto)                                  # Une palabras cortadas por guión o símbolo de silabeo OCR
    texto = re.sub(r"[^a-z0-9áéíóúüñ¡!¿?.,;():«»—+ſꝛũ'\s]", ' ', texto)  # Elimina caracteres que no aportan señal al modelo
    return re.sub(r'\s+', ' ', texto).strip()                              # Normaliza espacios múltiples

def clean_lr(texto):
    # Limpieza para la LogisticRegression: igual que el SVC pero conservando la ñ
    if not isinstance(texto, str): return ""
    texto = texto.lower()
    texto = re.sub(r'[-¬]\s*', '', texto)                                  # Une palabras cortadas por guión o símbolo de silabeo OCR
    texto = re.sub(r"[^a-z0-9áéíóúüñ¡!¿?.,;():«»—+ſꝛũ'\s]", ' ', texto)  # Elimina caracteres que no aportan señal al modelo
    return re.sub(r'\s+', ' ', texto).strip()                              # Normaliza espacios múltiples

def extraer_features_rf(texto):
    # Texto inválido o vacío — retorna ceros para no romper el pipeline
    if not isinstance(texto, str) or len(texto) == 0:
        return [0] * (N_FEATURES_ESTILO + len(TODOS_SIMBOLOS))

    palabras_lista  = texto.split()                                          # Separamos por espacios para tener la lista de palabras
    n_palabras      = max(len(palabras_lista), 1)                            # Cuántas palabras hay (el max evita dividir por cero después)
    n_caracteres    = len(texto)                                             # Longitud total del texto, espacios incluidos
    n_espacios      = max(texto.count(' '), 1)                               # Cuántos espacios hay (mínimo 1 para los ratios)
    n_saltos_linea  = texto.count('\n')                                      # Saltos de línea — dan forma al layout de la página
    total_letras    = max(len(re.findall(r'[a-zA-ZáéíóúüñÁÉÍÓÚÜÑ]', texto)), 1)  # Solo letras, sin números ni símbolos
    texto_lower     = texto.lower()                                          # Versión en minúsculas para búsquedas case-insensitive
    mayusculas_count         = sum(1 for c in texto if c.isupper())          # Cuántas letras están en mayúscula
    vocales_acentuadas_count = len(re.findall(r'[áéíóúÁÉÍÓÚ]', texto))     # Cuántas vocales llevan tilde
    frecuencias              = Counter(texto)                                 # Con qué frecuencia aparece cada carácter
    entropia = -sum((f / n_caracteres) * math.log2(f / n_caracteres)        # Entropía de Shannon sobre los caracteres del texto
                    for f in frecuencias.values()) if n_caracteres > 0 else 0  # Si está vacío, entropía cero
    freq_palabras   = Counter(palabras_lista)                                # Con qué frecuencia aparece cada palabra
    oraciones_lista = re.split(r'[.!?]+', texto)                            # Partimos en oraciones por los signos de puntuación fuerte
    n_oraciones     = max(len(oraciones_lista), 1)                           # Cuántas oraciones resultaron
    conteo_e   = len(re.findall(r'\be\b', texto_lower))                     # Veces que aparece 'e' como conjunción (forma arcaica de 'y')
    conteo_y   = len(re.findall(r'\by\b', texto_lower))                     # Veces que aparece 'y' como conjunción (forma moderna)
    conteo_non = len(re.findall(r'\bnon\b', texto_lower))                   # Veces que aparece 'non' (negación medieval, del latín)
    conteo_no  = len(re.findall(r'\bno\b', texto_lower))                    # Veces que aparece 'no' (negación moderna)
    conteo_i   = len(re.findall(r'\bi\b', texto_lower))                     # Veces que aparece 'i' conjuntiva (la 'y' del español medieval)

    features_estilo = [
        n_caracteres / n_palabras,                                           # Promedio de caracteres por palabra — palabras más largas sugieren latín o tecnicismos
        n_palabras,                                                          # Longitud del documento en palabras
        len(re.findall(r'[^a-zA-Z0-9áéíóúüñÁÉÍÓÚÜÑ\s]', texto)) / n_palabras,  # Cuántos símbolos y puntuación hay por cada palabra
        mayusculas_count / n_palabras,                                       # Mayúsculas por palabra — los textos latinos y medievales las usan mucho más
        max(len(re.findall(r'[.!?]+', texto)), 1) / n_palabras,             # Qué tan seguido aparecen los signos de cierre de oración
        np.mean([len(w) for w in palabras_lista]),                          # Longitud media de las palabras
        len(re.findall(r' {2,}', texto)) / n_espacios,                      # Proporción de dobles espacios — señal frecuente de OCR ruidoso
        len(PALABRAS_ARCAICAS.findall(texto_lower)) / n_palabras,           # Qué tan cargado está el texto de vocabulario arcaico español
        len(PALABRAS_LATINAS.findall(texto_lower)) / n_palabras,            # Cuántas palabras tienen terminaciones latinas
        vocales_acentuadas_count / total_letras,                            # Densidad de tildes — aumenta conforme se moderniza la ortografía
        len(re.findall(r'[¡¿]', texto)) / n_palabras,                       # Signos de apertura españoles — aparecen más en textos del s.XVIII en adelante
        len(re.findall(r'[«»]', texto)) / n_palabras,                       # Comillas angulares — propias de la imprenta clásica española
        len(re.findall(r'-', texto)) / n_palabras,                          # Guiones — tanto tipográficos como cortes de sílaba del OCR
        len(re.findall(r'[\^*|&£><@#~\\`{}[\]=]', texto)) / n_palabras,    # Caracteres extraños que el OCR introduce al escanear textos viejos
        entropia,                                                            # A más variedad de caracteres, mayor entropía — los textos medievales con ortografía libre la tienen alta
        len(set(palabras_lista)) / n_palabras,                              # Qué proporción de las palabras son únicas — mide riqueza de vocabulario
        sum(1 for w in palabras_lista if len(w) <= 3) / n_palabras,         # Cuántas palabras son muy cortas — artículos, preposiciones y partículas
        texto.count(' ') / n_palabras,                                      # Espacios totales por palabra — captura irregularidades del OCR
        n_saltos_linea / n_palabras,                                         # Qué tan fragmentado está el texto en líneas
        texto.count('\t') / n_palabras,                                      # Tabulaciones — casi nunca aparecen, pero cuando lo hacen son señal de formato especial
        n_caracteres / (n_saltos_linea + 1),                                 # Cuántos caracteres tiene cada línea en promedio
        len(re.findall(r'-\s*\n', texto)) / n_palabras,                     # Palabras cortadas con guión al final de línea — muy común en textos impresos escaneados
        sum(1 for c in texto if c.isdigit()) / n_palabras,                  # Densidad de números — más alta en textos jurídicos, científicos y modernos
        len([m for m in NUMEROS_ROMANOS.findall(texto) if len(m[0]) >= 2]) / n_palabras,  # Números romanos de al menos 2 caracteres (filtramos el 'I' suelto que da falsos positivos)
        len(ABREVIATURAS.findall(texto_lower)) / n_palabras,                # Palabras cortas seguidas de punto — las abreviaturas eran muy frecuentes en textos impresos antiguos
        len(U_COMO_V.findall(texto_lower)) / n_palabras,                    # 'v' usada como vocal — la confusión u/v es característica del español pre-moderno
        len(J_POR_X.findall(texto_lower)) / n_palabras,                     # Palabras donde convivían j y x (dixo/dijo) — rastro de la ortografía renacentista
        len(MAYUS_MID_ORACION.findall(texto)) / n_palabras,                 # Mayúsculas en mitad de oración — nombres propios o latinismos intercalados
        (len(re.findall(r'\n\s*\n', texto)) + 1) / n_palabras,             # Cuántos párrafos hay — el +1 cuenta también el primero
        len(CONECTORES_MODERNOS.findall(texto_lower)) / n_palabras,         # Conectores discursivos modernos — 'sin embargo', 'es decir', etc. son señal del s.XVIII-XIX
        len(VERBOS_ARCAICOS.findall(texto_lower)) / n_palabras,             # Verbos con desinencias medievales como -edes o -ades
        conteo_e / (conteo_e + conteo_y + 1),                               # Cuánto se usa 'e' frente a 'y' — cuanto más alto, más antiguo el texto
        conteo_non / (conteo_non + conteo_no + 1),                          # Cuánto se usa 'non' frente a 'no' — la negación latina fue cayendo en desuso
        len(re.findall(r'\bla\b', texto_lower)) / n_palabras,               # Frecuencia del artículo 'la' — señal sutil de registro y género
        len(CHARS_AISLADOS.findall(texto)) / n_palabras,                    # Consonantes sueltas sin contexto — basura típica del OCR en manuscritos
        len(re.findall(r'[bcdfghjklmnpqrstvwxyzBCDFGHJKLMNPQRSTVWXYZ]{3,}', texto)) / n_palabras,  # Racimos de consonantes — pueden ser latín, abreviaturas o artefactos del escáner
        sum(1 for f in freq_palabras.values() if f == 1) / n_palabras,      # Hapax legomena — palabras que solo aparecen una vez; más abundantes cuando la ortografía no estaba estandarizada
        sum(1 for w in palabras_lista if len(w) >= 8) / n_palabras,         # Palabras largas (8+ chars) — los textos renacentistas y latinos están llenos de ellas
        sum(1 for o in oraciones_lista if len(o.split()) > 30) / n_oraciones,  # Proporción de oraciones muy largas — el Siglo de Oro construía cláusulas interminables
        vocales_acentuadas_count / max(mayusculas_count, 1),                # Ratio tildes/mayúsculas — en textos muy antiguos hay pocas tildes y muchas mayúsculas latinas
        texto_lower.count('q') / total_letras,                              # Densidad de 'q' — los textos arcaicos la usaban en 'quando', 'qual', 'quatro'
        len(F_INICIAL.findall(texto_lower)) / n_palabras,                   # Palabras con f- inicial donde hoy va h- — huella del cambio f→h que ocurrió en el s.XV-XVI
        conteo_i / (conteo_i + conteo_y + 1),                               # Cuánto se usa 'i' frente a 'y' — antes de la RAE, la conjunción copulativa era 'i'
    ]

    return features_estilo + [texto.count(s) / n_palabras for s in TODOS_SIMBOLOS]

# ═════════════════════════════════════════════════════════════════════════════
# 1. CARGA Y PREPARACIÓN
# ═════════════════════════════════════════════════════════════════════════════
data      = pd.read_csv("../../data/train.csv")   # Dataset de entrenamiento con columnas 'text' y 'decade'
data_test = pd.read_csv("../../data/eval.csv")    # Dataset de evaluación — solo tiene 'id' y 'text'

# Aplicamos la limpieza correspondiente a cada modelo
data["text_svc"]      = data["text"].apply(clean_svc)
data["text_lr"]       = data["text"].apply(clean_lr)
data_test["text_svc"] = data_test["text"].apply(clean_svc)
data_test["text_lr"]  = data_test["text"].apply(clean_lr)

# El RF trabaja sobre el texto crudo (sin limpiar) para preservar mayúsculas y símbolos OCR
feat_train = np.array(data["text"].apply(extraer_features_rf).tolist(),      dtype=np.float32)
feat_test  = np.array(data_test["text"].apply(extraer_features_rf).tolist(), dtype=np.float32)

y = data["decade"]   # Variable objetivo: la década del texto

# Mismo split para los 3 modelos — garantiza que comparan sobre las mismas muestras
X_svc_train, X_svc_val, y_train, y_val = train_test_split(
    data["text_svc"], y, test_size=0.2, random_state=42, stratify=y
)
X_lr_train, X_lr_val, _, _ = train_test_split(
    data["text_lr"], y, test_size=0.2, random_state=42, stratify=y
)
feat_tr, feat_val = train_test_split(
    feat_train, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_svc_train)} | Val: {len(X_svc_val)}")


# ═════════════════════════════════════════════════════════════════════════════
# 2. TF-IDF
# ═════════════════════════════════════════════════════════════════════════════
# Usamos char_wb (n-gramas de caracteres) porque captura mejor las variaciones
# ortográficas históricas que los n-gramas de palabras
tfidf_svc = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4),
                             max_df=0.85, min_df=2, max_features=150000,
                             lowercase=False, sublinear_tf=True)
tfidf_lr  = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4),
                             max_df=0.85, min_df=2, max_features=150000,
                             lowercase=False, sublinear_tf=True)

X_svc_tr_v  = tfidf_svc.fit_transform(X_svc_train)   # Aprende el vocabulario y transforma el train del SVC
X_svc_val_v = tfidf_svc.transform(X_svc_val)          # Solo transforma — no aprende nada nuevo del val
X_lr_tr_v   = tfidf_lr.fit_transform(X_lr_train)      # Aprende el vocabulario y transforma el train del LR
X_lr_val_v  = tfidf_lr.transform(X_lr_val)            # Solo transforma — no aprende nada nuevo del val


# ═════════════════════════════════════════════════════════════════════════════
# 3. SELECCIÓN DE FEATURES PARA EL RF (umbral=0.0013)
#
#    El RF no usa todas las 102 features directamente — primero entrena un RF
#    ligero de 500 árboles para calcular la importancia Gini de cada feature,
#    luego descarta las que aportan menos del 0.13% de importancia.
#    Esto reduce el ruido y mejora la generalización del modelo definitivo.
# ═════════════════════════════════════════════════════════════════════════════
scaler_sel = StandardScaler()          # Estandariza las features antes del RF inicial
rf_sel     = RandomForestClassifier(n_estimators=500, max_features='sqrt',
                                     min_samples_leaf=2, class_weight='balanced',
                                     random_state=42, n_jobs=-1)
rf_sel.fit(scaler_sel.fit_transform(feat_tr), y_train)

imp_df = pd.DataFrame({'feature': FEATURE_NAMES,
                        'importancia': rf_sel.feature_importances_}
                      ).sort_values('importancia', ascending=False)

# Nos quedamos solo con las features que superan el umbral
features_retenidas = imp_df[imp_df['importancia'] >= 0.0013]['feature'].tolist()
indices_rf         = [FEATURE_NAMES.index(f) for f in features_retenidas]   # Índices de columnas a conservar

# Aplicamos el mismo corte a todos los conjuntos de datos
feat_tr_r    = feat_tr[:,    indices_rf]
feat_val_r   = feat_val[:,   indices_rf]
feat_train_r = feat_train[:, indices_rf]   
feat_test_r  = feat_test[:,  indices_rf]   

scaler_rf  = StandardScaler()
X_rf_tr_s  = scaler_rf.fit_transform(feat_tr_r)
X_rf_val_s = scaler_rf.transform(feat_val_r)
print(f"Features RF retenidas: {len(features_retenidas)}")


# ═════════════════════════════════════════════════════════════════════════════
# 4. ENTRENAMIENTO DE LOS 3 MODELOS
#
#    Cada modelo opera sobre una representación distinta del texto:
#    - LinearSVC y LogisticRegression → TF-IDF de char n-gramas (señal léxica)
#    - Random Forest → features estilométricas manuales (señal estructural)
#    Esta diversidad es lo que hace útil el ensamble.
# ═════════════════════════════════════════════════════════════════════════════

# LinearSVC no tiene predict_proba nativo — lo calibramos con sigmoid para
# que devuelva probabilidades y pueda participar en el soft voting
modelo_svc = CalibratedClassifierCV(
    LinearSVC(C=0.5, class_weight='balanced', penalty='l2',
              random_state=42, max_iter=5000, dual=False),
    method='sigmoid', cv=3
)
modelo_svc.fit(X_svc_tr_v, y_train)
acc_svc = accuracy_score(y_val, modelo_svc.predict(X_svc_val_v))
print(f"  LinearSVC          : {acc_svc:.4f}")

modelo_lr = LogisticRegression(
    C=6, class_weight='balanced', penalty='l2',
    solver='lbfgs', random_state=42, max_iter=5000
)
modelo_lr.fit(X_lr_tr_v, y_train)
acc_lr = accuracy_score(y_val, modelo_lr.predict(X_lr_val_v))
print(f"  LogisticRegression : {acc_lr:.4f}")

# Random Forest — 700 árboles es el número que maximizó la métrica de validación
modelo_rf = RandomForestClassifier(
    n_estimators=700, max_features='sqrt', min_samples_leaf=2,
    max_depth=None, class_weight='balanced', random_state=42, n_jobs=-1
)
modelo_rf.fit(X_rf_tr_s, y_train)
acc_rf = accuracy_score(y_val, modelo_rf.predict(X_rf_val_s))
print(f"  Random Forest      : {acc_rf:.4f}")


# ═════════════════════════════════════════════════════════════════════════════
# 5. SOFT VOTING — BÚSQUEDA DE PESOS ÓPTIMOS
#
#    En soft voting cada modelo aporta su vector de probabilidades por clase.
#    La predicción final es: argmax(w1*prob_svc + w2*prob_lr + w3*prob_rf)
#    Buscamos en rejilla los pesos w1, w2, w3 que maximizan el val accuracy,
#    con la restricción de que w1 + w2 + w3 = 1.
# ═════════════════════════════════════════════════════════════════════════════

# Obtenemos las probabilidades de cada modelo sobre el set de validación
prob_svc = modelo_svc.predict_proba(X_svc_val_v)   # shape: (n_val, 39 clases)
prob_lr  = modelo_lr.predict_proba(X_lr_val_v)     # shape: (n_val, 39 clases)
prob_rf  = modelo_rf.predict_proba(X_rf_val_s)     # shape: (n_val, 39 clases)
clases   = modelo_svc.classes_                     # Array con los 39 nombres de clase (décadas)

mejor_acc_soft = 0
mejor_pesos    = (0.33, 0.33, 0.34)

# Recorremos todas las combinaciones válidas donde w1+w2+w3=1
for w1 in np.arange(0.0, 1.05, 0.05):
    for w2 in np.arange(0.0, 1.05, 0.05):
        w3 = round(1.0 - w1 - w2, 4)
        if w3 < 0 or w3 > 1: continue
        acc = accuracy_score(y_val, clases[np.argmax(w1*prob_svc + w2*prob_lr + w3*prob_rf, axis=1)])
        if acc > mejor_acc_soft:
            mejor_acc_soft = acc
            mejor_pesos    = (round(w1, 2), round(w2, 2), round(w3, 2))

w_svc_opt, w_lr_opt, w_rf_opt = mejor_pesos
print(f"  w_svc={w_svc_opt} | w_lr={w_lr_opt} | w_rf={w_rf_opt}")
print(f"  Val Accuracy óptimo: {mejor_acc_soft:.4f}")


# ═════════════════════════════════════════════════════════════════════════════
# 6. REENTRENAMIENTO SOBRE EL 100% Y PREDICCIONES
#
#    Ahora, reentrenamos cada modelo con TODOS los datos de train para que aprenda
#    de la mayor cantidad posible de ejemplos antes de predecir sobre eval.
# ═════════════════════════════════════════════════════════════════════════════

# Nuevos vectorizadores que aprenden sobre TODO el train (no solo el 80%)
tfidf_svc_f = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4),
                               max_df=0.85, min_df=2, max_features=150000,
                               lowercase=False, sublinear_tf=True)
tfidf_lr_f  = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4),
                               max_df=0.85, min_df=2, max_features=150000,
                               lowercase=False, sublinear_tf=True)

X_full_svc = tfidf_svc_f.fit_transform(data["text_svc"])   # Aprende vocabulario completo
X_full_lr  = tfidf_lr_f.fit_transform(data["text_lr"])     # Aprende vocabulario completo

# Reentrenamos el SVC sobre el 100% con los mismos parámetros ganadores
svc_f = CalibratedClassifierCV(
    LinearSVC(C=0.5, class_weight='balanced', random_state=42,
              max_iter=5000, dual=False), method='sigmoid', cv=3)
svc_f.fit(X_full_svc, y)

# Reentrenamos la LR sobre el 100% con los mismos parámetros ganadores
lr_f = LogisticRegression(C=6, class_weight='balanced',
                           solver='lbfgs', random_state=42, max_iter=5000)
lr_f.fit(X_full_lr, y)

# Reentrenamos el RF sobre el 100% — aplicamos el mismo scaler sobre las features retenidas
scaler_rf_f = StandardScaler()
X_full_rf_s = scaler_rf_f.fit_transform(feat_train_r)   # feat_train_r ya tiene solo las features seleccionadas
X_test_rf_s = scaler_rf_f.transform(feat_test_r)

rf_f = RandomForestClassifier(
    n_estimators=700, max_features='sqrt', min_samples_leaf=2,
    max_depth=None, class_weight='balanced', random_state=42, n_jobs=-1)
rf_f.fit(X_full_rf_s, y)

# Soft voting con los pesos óptimos encontrados en la sección 5
prob_svc_test = svc_f.predict_proba(tfidf_svc_f.transform(data_test["text_svc"]))
prob_lr_test  = lr_f.predict_proba(tfidf_lr_f.transform(data_test["text_lr"]))
prob_rf_test  = rf_f.predict_proba(X_test_rf_s)

prob_final   = w_svc_opt * prob_svc_test + w_lr_opt * prob_lr_test + w_rf_opt * prob_rf_test
predicciones = svc_f.classes_[np.argmax(prob_final, axis=1)]   # La clase con mayor probabilidad combinada

pd.DataFrame({"id": data_test["id"], "answer": predicciones}).to_csv(
    "winner_model_ensamble.csv", index=False
)


# ═════════════════════════════════════════════════════════════════════════════
# 7. GUARDAR EL ENSAMBLE EN JOBLIB
# ═════════════════════════════════════════════════════════════════════════════
ensamble = {
    "modelo_svc":  svc_f,          # LinearSVC calibrado entrenado sobre el 100%
    "tfidf_svc":   tfidf_svc_f,    # Vectorizador TF-IDF del SVC con vocabulario completo
    "modelo_lr":   lr_f,           # LogisticRegression entrenada sobre el 100%
    "tfidf_lr":    tfidf_lr_f,     # Vectorizador TF-IDF del LR con vocabulario completo
    "modelo_rf":   rf_f,           # Random Forest entrenado sobre el 100%
    "scaler_rf":   scaler_rf_f,    # StandardScaler ajustado sobre el 100% de features del RF
    "indices_rf":  indices_rf,     # Índices de las ~69 features retenidas tras el corte Gini
    "w_svc":       w_svc_opt,      # Peso óptimo del SVC en el soft voting
    "w_lr":        w_lr_opt,       # Peso óptimo del LR en el soft voting
    "w_rf":        w_rf_opt,       # Peso óptimo del RF en el soft voting
}
joblib.dump(ensamble, "ensamble_final_serializado.joblib")
print(f"\n  ══════════════════════════════════════════")
print(f"  RESUMEN FINAL")
print(f"  ══════════════════════════════════════════")
print(f"  LinearSVC           : {acc_svc:.4f}")
print(f"  LogisticRegression  : {acc_lr:.4f}")
print(f"  Random Forest       : {acc_rf:.4f}")
print(f"  Soft Voting óptimo  : {mejor_acc_soft:.4f}")
print(f"  Pesos               : w_svc={w_svc_opt} | w_lr={w_lr_opt} | w_rf={w_rf_opt}")
print(f"  ══════════════════════════════════════════")

Train: 25122 | Val: 6281
Features RF retenidas: 69
  LinearSVC          : 0.2710


/Users/jhonvega/Documents/universidad/Aprendizaje de Maquina/project/taller_nlp/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


  LogisticRegression : 0.2692
  Random Forest      : 0.2388
  w_svc=0.4 | w_lr=0.0 | w_rf=0.6
  Val Accuracy óptimo: 0.3136

  ══════════════════════════════════════════
  RESUMEN FINAL
  ══════════════════════════════════════════
  LinearSVC           : 0.2710
  LogisticRegression  : 0.2692
  Random Forest       : 0.2388
  Soft Voting óptimo  : 0.3136
  Pesos               : w_svc=0.4 | w_lr=0.0 | w_rf=0.6
  ══════════════════════════════════════════


# Cargar y obtener las predicciones del modelo serializado (ensamble_final_serializado.joblib)

In [2]:
import joblib
import re
import math
import numpy as np
import pandas as pd
from collections import Counter

#  Funciones de preprocesamiento 

def clean_svc(texto):
    if not isinstance(texto, str): return ""
    texto = texto.lower().replace('ñ', 'n')
    texto = re.sub(r'[-¬]\s*', '', texto)
    texto = re.sub(r"[^a-z0-9áéíóúüñ¡!¿?.,;():«»—+ſꝛũ'\s]", ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()

# (pega aquí la función extraer_features_rf completa con todos sus regex)

# Carga de datos

data_test = pd.read_csv("../../data/eval.csv")   # ajusta la ruta si es necesario

# Carga del modelo 

ensamble = joblib.load("ensamble_final_serializado.joblib")

# Preprocesamiento 

data_test["text_svc"] = data_test["text"].apply(clean_svc)
feat_test = np.array(data_test["text"].apply(extraer_features_rf).tolist(), dtype=np.float32)

# Predicción

prob_svc = ensamble["modelo_svc"].predict_proba(
    ensamble["tfidf_svc"].transform(data_test["text_svc"])
)
prob_rf = ensamble["modelo_rf"].predict_proba(
    ensamble["scaler_rf"].transform(feat_test[:, ensamble["indices_rf"]])
)

prob_final   = ensamble["w_svc"] * prob_svc + ensamble["w_rf"] * prob_rf
predicciones = ensamble["modelo_svc"].classes_[np.argmax(prob_final, axis=1)]

# Exportar 

pd.DataFrame({
    "id":     data_test["id"],
    "answer": predicciones
}).to_csv("winner_model_loaded.csv", index=False)
